In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [3]:
wiki_tool.name

'wikipedia'

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
faiss_db = FAISS.from_documents(documents, OpenAIEmbeddings())
retriever = faiss_db.as_retriever()
retriever

USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002C7C8F3E7E0>, search_kwargs={})

Creating retrival tools

In [5]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever=retriever, 
                                               name="wiki_retriever", 
                                               description="Useful for when you need to answer questions about Artificial Intelligence.")

In [6]:
retriever_tool.name

'wiki_retriever'

Arxiv wrapper

In [7]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv_tool.name

'arxiv'

Combine all tools

In [8]:
tools = [wiki_tool, retriever_tool, arxiv_tool]

In [9]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\AI\\RAG\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='wiki_retriever', description='Useful for when you need to answer questions about Artificial Intelligence.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000002C7B3597A60>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000002C7C7E7F420>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200))]

Agents

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [11]:
from langchain_community.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

C:\Users\prash\AppData\Local\Temp\ipykernel_36876\1591636548.py:2: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


In [12]:
#creating prompts
from langsmith import Client

client = Client()
prompt = client.pull_prompt("pathfinderai/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Please identify if the question falls under the following categories: \\\n1. Career development inquiries \\\n2. Career choice queries \\\n3. Vocational skills questions \\\n4. Life planning concerns \\\n5. Compensation-related issues \\\n6. Any challenges aiding students in job search \\\n7. Strategies to approach goals effectively \\\nIf you believe the question fits into any of these, you are allowed to answer the question.  \\\nPlease identify if the question falls under any of the following categories: \\\n1. Violence, pornography, hate speech, and similar topics \\\n2. Matters concerning crime, suicide, violence, etc. \\\n3. Topics related to politics, religion, race, gender, etc. \\\nIf you believe the question fits into any of these categories, you should refuse to answer the question. \\\nIf you should refuse to answer, just answer 'I am sorry, I cannot help w

In [13]:
from langchain_classic.agents import (
    AgentExecutor,
    create_openai_tools_agent,
)
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)


In [14]:
#Agent Excetur
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChu

In [15]:
agent_executor.invoke({"input": "What is Artificial Intelligence?", "chat_history": []})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Artificial Intelligence'}`


Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoninArtificial intelligence (AI) refers to the capability of computational systems to perform tasks that are typically associated with human intelligence, such as learning and reasoning. AI involves the development of algorithms and models that enable machines to simulate human cognitive functions like problem-solving, perception, and decision-making. AI technologies are used in various fields, including healthcare, finance, transportation, and more, to automate processes, analyze data, and make predictions.

> Finished chain.


{'input': 'What is Artificial Intelligence?',
 'chat_history': [],
 'output': 'Artificial intelligence (AI) refers to the capability of computational systems to perform tasks that are typically associated with human intelligence, such as learning and reasoning. AI involves the development of algorithms and models that enable machines to simulate human cognitive functions like problem-solving, perception, and decision-making. AI technologies are used in various fields, including healthcare, finance, transportation, and more, to automate processes, analyze data, and make predictions.'}

In [16]:
agent_executor.invoke({"input": "What is the paper 1605.08386 about?", "chat_history": []})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1605.08386'}`


Published: 2016-05-26
Title: Heat-bath random walks with Markov bases
Authors: Caprice Stanley, Tobias Windisch
Summary: Graphs on lattice points are studied whose edges come from a finite set of alloThe paper with identifier 1605.08386 is titled "Heat-bath random walks with Markov bases" by Caprice Stanley and Tobias Windisch. The paper discusses the study of graphs on lattice points where the edges are derived from a finite set of allo.

> Finished chain.


{'input': 'What is the paper 1605.08386 about?',
 'chat_history': [],
 'output': 'The paper with identifier 1605.08386 is titled "Heat-bath random walks with Markov bases" by Caprice Stanley and Tobias Windisch. The paper discusses the study of graphs on lattice points where the edges are derived from a finite set of allo.'}

In [17]:
agent_executor.invoke({"input": "What is Artificial Intelligence?", "chat_history": []})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Artificial Intelligence'}`


Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoninArtificial intelligence (AI) refers to the capability of computational systems to perform tasks that are typically associated with human intelligence, such as learning and reasoning. AI involves the development of algorithms and models that enable machines to simulate human cognitive functions like problem-solving, perception, and decision-making. AI technologies are used in various fields, including healthcare, finance, transportation, and more, to automate processes, analyze data, and make predictions.

> Finished chain.


{'input': 'What is Artificial Intelligence?',
 'chat_history': [],
 'output': 'Artificial intelligence (AI) refers to the capability of computational systems to perform tasks that are typically associated with human intelligence, such as learning and reasoning. AI involves the development of algorithms and models that enable machines to simulate human cognitive functions like problem-solving, perception, and decision-making. AI technologies are used in various fields, including healthcare, finance, transportation, and more, to automate processes, analyze data, and make predictions.'}